# Notebook 06 — Final Dataset Assembly

Reads per-event strandings data with weather, moon, and plankton features, then aggregates to weekly counts per region for modeling.

**Reads:** `strandings.parquet`, `strandings_with_weather.parquet`, `strandings_with_moon.parquet`, `plankton_imputed_lookup.parquet`  
**Writes:** `data/processed/final_dataset.parquet`

**Prerequisites:** Fixes F1–F3 must be complete before running this notebook.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from se_coast_strandings.contextual_data.plankton_abundance import (
    assign_region, DEFAULT_REGIONS
)
from se_coast_strandings.transformations import (
    make_dt_col, make_cyclic, make_cyclic_season
)
from se_coast_strandings.contextual_data.lunar_phases import add_moon_features

PROCESSED_DIR = Path("../data/processed")

## Load and verify base strandings

In [ ]:
base = pd.read_parquet(PROCESSED_DIR / "strandings.parquet")
base["mms_observation_dt"] = make_dt_col(
    base["Day of Observation"],
    base["Month of Observation"],
    base["Year of Observation"],
)
base = base.dropna(subset=["mms_observation_dt", "Latitude", "Longitude"])
base = base.reset_index(drop=True)
print(f"Base strandings: {len(base)} rows")

## Join weather and moon features

In [ ]:
weather = pd.read_parquet(PROCESSED_DIR / "strandings_with_weather.parquet")
weather_cols = [c for c in weather.columns if c not in base.columns]
base = base.join(weather[weather_cols])
assert len(base) == len(weather), "Row count mismatch after weather join"

moon = pd.read_parquet(PROCESSED_DIR / "strandings_with_moon.parquet")
base[["moon_age", "moon_phase"]] = moon[["moon_age", "moon_phase"]]
print(f"After joins: {len(base)} rows, {len(base.columns)} columns")

## Assign regions

In [ ]:
base["region"] = assign_region(base["Latitude"])
base = base.dropna(subset=["region"])
print(f"After region assignment: {len(base)} rows")
print(base["region"].value_counts())

## Build cyclic features and snap to week

In [ ]:
base["week_start"] = base["mms_observation_dt"].dt.to_period("W").dt.start_time

base["month_sin"], base["month_cos"] = make_cyclic(
    base["mms_observation_dt"].dt.month, 12, name="month"
)
base["dayofyear_sin"], base["dayofyear_cos"] = make_cyclic(
    base["mms_observation_dt"].dt.dayofyear, 365, name="dayofyear"
)
base["season_sin"], base["season_cos"] = make_cyclic_season(
    base["mms_observation_dt"], name="season"
)

## Aggregate to weekly counts per region

In [ ]:
day0_max = "temperature_2m_max_0_days_prior"
day0_min = "temperature_2m_min_0_days_prior"
day1_max = "temperature_2m_max_1_days_prior"

if day1_max in base.columns:
    base["temp_delta_day0"] = base[day0_max] - base[day1_max]

agg_dict = {
    "NMFS Ref #": "count",
    day0_max: ["mean", "max"],
    day0_min: "mean",
    "moon_age": "mean",
    "temp_delta_day0": "mean",
    "month_sin": "first", "month_cos": "first",
    "dayofyear_sin": "first", "dayofyear_cos": "first",
    "season_sin": "first", "season_cos": "first",
}

# Only include columns that exist
agg_dict = {k: v for k, v in agg_dict.items() if k in base.columns}

weekly = base.groupby(["week_start", "region"]).agg(agg_dict).reset_index()

# Flatten MultiIndex columns
weekly.columns = [
    "_".join(filter(None, c)).strip("_") if isinstance(c, tuple) else c
    for c in weekly.columns
]
weekly = weekly.rename(columns={"NMFS Ref #_count": "stranding_count"})
print(f"Weekly aggregated: {len(weekly)} rows")

## Fill zero-stranding weeks (CRITICAL)

Weeks with no strandings must be explicitly represented as `stranding_count=0`.
Without this step, the model never sees low-risk periods and cannot learn the baseline rate.

In [ ]:
all_weeks = pd.date_range(
    start=base["week_start"].min(),
    end=base["week_start"].max(),
    freq="W-MON",
)
all_regions = [label for label, _, _ in DEFAULT_REGIONS]

full_index = pd.MultiIndex.from_product(
    [all_weeks, all_regions], names=["week_start", "region"]
)
full_grid = pd.DataFrame(index=full_index).reset_index()

weekly = full_grid.merge(weekly, on=["week_start", "region"], how="left")
weekly["stranding_count"] = weekly["stranding_count"].fillna(0).astype(int)
print(f"After zero-fill: {len(weekly)} rows ({len(all_weeks)} weeks \u00d7 {len(all_regions)} regions)")

## Join plankton density lookup

In [ ]:
plankton = pd.read_parquet(PROCESSED_DIR / "plankton_imputed_lookup.parquet")
plankton = plankton.rename(columns={"ds": "week_start", "yhat": "plankton_density"})
plankton["week_start"] = pd.to_datetime(plankton["week_start"])

weekly = weekly.merge(
    plankton[["week_start", "region", "plankton_density"]],
    on=["week_start", "region"], how="left"
)
print(f"Plankton density NaN count: {weekly['plankton_density'].isna().sum()}")

## Recompute features for all weeks + forward-fill weather

In [ ]:
weekly = weekly.sort_values(["region", "week_start"]).reset_index(drop=True)

# Recompute cyclic and moon features for ALL weeks (including zero-stranding)
weekly["month_sin"], weekly["month_cos"] = make_cyclic(
    weekly["week_start"].dt.month, 12, name="month"
)
weekly["dayofyear_sin"], weekly["dayofyear_cos"] = make_cyclic(
    weekly["week_start"].dt.dayofyear, 365, name="dayofyear"
)
weekly["season_sin"], weekly["season_cos"] = make_cyclic_season(
    weekly["week_start"], name="season"
)
weekly = add_moon_features(weekly, date_col="week_start")

# Forward-fill weather within each region for zero-stranding weeks
weather_feat_cols = [c for c in weekly.columns if c.startswith("temperature_2m") or c == "temp_delta_day0_mean"]
weekly[weather_feat_cols] = (
    weekly.groupby("region")[weather_feat_cols]
    .transform(lambda g: g.ffill().bfill())
)
print(f"Weather NaN after fill: {weekly[weather_feat_cols].isna().sum().sum()}")

## Add lag features

In [ ]:
weekly["stranding_count_lag_1"] = weekly.groupby("region")["stranding_count"].shift(1)
weekly["stranding_count_lag_52"] = weekly.groupby("region")["stranding_count"].shift(52)
# Drop first 52 weeks per region (insufficient lag history)
weekly = weekly.dropna(subset=["stranding_count_lag_52"]).reset_index(drop=True)
print(f"After lag drop: {len(weekly)} rows")

## Encode region and save

In [ ]:
region_dummies = pd.get_dummies(weekly["region"], prefix="region", drop_first=False)
weekly = pd.concat([weekly, region_dummies], axis=1)

# Schema validation
REQUIRED_COLS = [
    "week_start", "region", "stranding_count",
    "month_sin", "month_cos", "dayofyear_sin", "dayofyear_cos",
    "season_sin", "season_cos", "moon_age", "plankton_density",
    "stranding_count_lag_1", "stranding_count_lag_52",
]
for col in REQUIRED_COLS:
    assert col in weekly.columns, f"Missing required column: {col}"

weekly.to_parquet(PROCESSED_DIR / "final_dataset.parquet", index=False)
print(f"Saved final_dataset.parquet: {len(weekly):,} rows \u00d7 {len(weekly.columns)} columns")
print(f"\nColumns: {list(weekly.columns)}")
weekly.head()